O objetivo principal é desenvolver as features relacionadas a método de pagamento \
1- GMV por tipo de pagamento - uma feature cada \
2- % Pedidos por meio de Pagamento \
3- Média quantidade de parcelas (excluindo à vista)

In [0]:
  SELECT
      SUM(CASE WHEN pg.vlPagamento IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100 AS pct_vlPagamento_null,
      SUM(CASE WHEN pg.descTipoPagamento IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100 AS pct_descTipoPagamento_null,
      SUM(CASE WHEN pd.dtPedido IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100 AS pct_dtPedido_null,
      SUM(CASE WHEN pg.nrParcelas  IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100 AS pct_nrParcelas_null
  FROM silver.olist.pagamento_pedido AS pg
      LEFT JOIN silver.olist.pedido AS pd
        ON pd.idPedido = pg.idPedido
 

pct_vlPagamento_null,pct_descTipoPagamento_null,pct_dtPedido_null,pct_nrParcelas_null
0.0,0.0,0.0,0.0


In [0]:
WITH info AS (
  SELECT
      SUM(pg.vlPagamento) AS total_pagto,
      pg.descTipoPagamento AS tp_pagamento,
      date_trunc('month', pd.dtPedido) AS mes_pedido
  FROM silver.olist.pagamento_pedido AS pg
      LEFT JOIN silver.olist.pedido AS pd
        ON pd.idPedido = pg.idPedido
  GROUP BY pg.descTipoPagamento, mes_pedido
)
SELECT
    mes_pedido,
    SUM(CASE WHEN tp_pagamento = 'credit_card' THEN total_pagto ELSE 0 END) AS cartao_credito,
    SUM(CASE WHEN tp_pagamento = 'debit_card' THEN total_pagto ELSE 0 END) AS cartao_debito,
    SUM(CASE WHEN tp_pagamento = 'vouncher' THEN total_pagto ELSE 0 END) AS vouncher,
    SUM(CASE WHEN tp_pagamento = 'not_defined' THEN total_pagto ELSE 0 END) AS not_defined
FROM info
GROUP BY mes_pedido
ORDER BY mes_pedido;


mes_pedido,cartao_credito,cartao_debito,vouncher,not_defined
2016-09-01T00:00:00Z,252.23999404907227,0.0,0.0,0.0
2016-10-01T00:00:00Z,48290.62004160881,241.72999572753906,0.0,0.0
2016-12-01T00:00:00Z,19.6200008392334,0.0,0.0,0.0
2017-01-01T00:00:00Z,109615.68001586199,743.5300025939941,0.0,0.0
2017-02-01T00:00:00Z,226753.55962496996,1510.3199882507324,0.0,0.0
2017-03-01T00:00:00Z,354488.93955922127,3592.7999897003174,0.0,0.0
2017-04-01T00:00:00Z,322087.7195917368,2790.0000038146973,0.0,0.0
2017-05-01T00:00:00Z,445047.31009730697,3023.4100017547607,0.0,0.0
2017-06-01T00:00:00Z,388268.2698506713,2424.169979095459,0.0,0.0
2017-07-01T00:00:00Z,454988.46965652704,2345.029983520508,0.0,0.0


Dúvidas 
1- O que significa cada tipo de pagamento? 
- vouncher
- not defined
- cartao de debito
- cartao de crédito 

2- 

In [0]:
WITH info AS (
  SELECT
      COUNT(DISTINCT pg.idPedido) AS pedidos_unicos,
      pg.descTipoPagamento AS tp_pagamento,
      date_trunc('month', pd.dtPedido) AS mes_pedido
  FROM silver.olist.pagamento_pedido AS pg
      LEFT JOIN silver.olist.pedido AS pd
        ON pd.idPedido = pg.idPedido
  GROUP BY pg.descTipoPagamento, mes_pedido
),
total_pedidos AS (
  SELECT
      mes_pedido,
      SUM(pedidos_unicos) AS total_pedidos
  FROM info
  GROUP BY mes_pedido
)
SELECT
    i.mes_pedido,
    ROUND(100.0 * SUM(CASE WHEN i.tp_pagamento = 'credit_card' THEN i.pedidos_unicos ELSE 0 END) / t.total_pedidos, 2) AS cartao_credito,
    ROUND(100.0 * SUM(CASE WHEN i.tp_pagamento = 'debit_card' THEN i.pedidos_unicos ELSE 0 END) / t.total_pedidos, 2) AS cartao_debito,
    ROUND(100.0 * SUM(CASE WHEN i.tp_pagamento = 'vouncher' THEN i.pedidos_unicos ELSE 0 END) / t.total_pedidos, 2) AS vouncher,
    ROUND(100.0 * SUM(CASE WHEN i.tp_pagamento = 'not_defined' THEN i.pedidos_unicos ELSE 0 END) / t.total_pedidos, 2) AS not_defined
FROM info i
JOIN total_pedidos t
  ON i.mes_pedido = t.mes_pedido
GROUP BY i.mes_pedido, t.total_pedidos
ORDER BY i.mes_pedido;

mes_pedido,cartao_credito,cartao_debito,vouncher,not_defined
2016-09-01T00:00:00Z,100.00,0.00,0.00,0.00
2016-10-01T00:00:00Z,76.90,0.61,0.00,0.00
2016-12-01T00:00:00Z,100.00,0.00,0.00,0.00
2017-01-01T00:00:00Z,70.89,1.10,0.00,0.00
2017-02-01T00:00:00Z,73.73,0.71,0.00,0.00
2017-03-01T00:00:00Z,72.97,1.13,0.00,0.00
2017-04-01T00:00:00Z,74.20,1.09,0.00,0.00
2017-05-01T00:00:00Z,74.44,0.79,0.00,0.00
2017-06-01T00:00:00Z,73.68,0.81,0.00,0.00
2017-07-01T00:00:00Z,74.13,0.53,0.00,0.00


In [0]:
WITH info AS (
  SELECT
    pg.idPedido, 
    max(nrParcelas) as qtd_parcelas,
    date_trunc('month', pd.dtPedido) AS mes_pedido
  FROM silver.olist.pagamento_pedido AS pg
  LEFT JOIN silver.olist.pedido AS pd
    ON pd.idPedido = pg.idPedido
  WHERE pg.descTipoPagamento = 'credit_card'
  GROUP BY  mes_pedido , pg.idPedido
)
SELECT
  round(avg(qtd_parcelas), 2) AS avg_parcelas,
  mes_pedido
FROM info
GROUP BY mes_pedido


avg_parcelas,mes_pedido
3.42,2018-06-01T00:00:00Z
3.8,2017-01-01T00:00:00Z
3.41,2018-03-01T00:00:00Z
3.33,2018-08-01T00:00:00Z
3.22,2018-01-01T00:00:00Z
3.66,2017-03-01T00:00:00Z
3.98,2017-07-01T00:00:00Z
2.0,2016-09-01T00:00:00Z
3.56,2017-10-01T00:00:00Z
3.52,2017-12-01T00:00:00Z


Dúvidas \
1- Só cartão de crédito possui mapgamento em parcelas? \
2- Como dentificar quando o pagamento é a vista?